# Week 1 Data Science Internship Assignment
**Mentor:** Laiba Sattar  
**Dataset:** Titanic Passenger Data  
**Student:** [Your Name]

---


## Section 1 — Dataset Introduction

I chose the **Titanic Passenger Data** from Kaggle. It contains information about 891 passengers who were aboard the RMS Titanic when it sank in April 1912.

**Why I chose it:**  
The Titanic dataset is a classic starting point for EDA and machine learning because it contains a rich mix of numeric columns (age, fare, family size), categorical columns (sex, class, embarkation port), and real-world messiness (missing values, outliers, and class imbalance). It also carries a compelling human story — making it easier to interpret findings in plain English.

**Columns include:**
- `Survived` — whether the passenger survived (0 = No, 1 = Yes)
- `Pclass` — ticket class (1 = First, 2 = Second, 3 = Third)
- `Name`, `Sex`, `Age` — personal details
- `SibSp`, `Parch` — number of siblings/spouses/parents/children aboard
- `Ticket`, `Fare` — ticketing info
- `Cabin`, `Embarked` — cabin number and port of embarkation


## Setup — Import Libraries & Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load the Titanic dataset directly from a public URL
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")


---
## Section 2 — The 7-Command First-Look Protocol


### C1 — `df.shape` — Dataset Dimensions

In [ ]:
df.shape

**Interpretation:**  
The dataset has **891 rows and 12 columns**. This is a medium-small dataset — large enough for meaningful EDA and classical ML algorithms (logistic regression, decision trees), but too small for deep learning without data augmentation. 891 passengers represent roughly 40% of those aboard, as many records were lost.


### C2 — `df.dtypes` — Data Types of Every Column

In [ ]:
df.dtypes

**Interpretation:**  
- `Age` is `float64` (not `int64`) — this is because missing values force Pandas to use floats (NaN is a float). If Age were fully complete, it would likely be integer.
- `Name`, `Sex`, `Ticket`, `Cabin`, `Embarked` are `object` (text). Some of these may need encoding before modelling.
- `Survived`, `Pclass`, `SibSp`, `Parch` are `int64` — discrete numeric columns. Note that `Survived` and `Pclass` are really *categorical*, even though they are stored as integers.
- No `datetime` columns here. The embarkation date is not recorded per-passenger.


### C3 — `df.info()` — Non-Null Counts, Types & Memory

In [ ]:
df.info()

**Interpretation:**  
Three columns have missing values:
- **`Age`**: 714 non-null out of 891 → **177 missing (19.9%)**. Significant but manageable — median/model-based imputation is common.
- **`Cabin`**: only 204 non-null out of 891 → **687 missing (77.1%)**. Nearly three-quarters of cabin data is gone. This column may be dropped or converted to a binary "has cabin info" feature.
- **`Embarked`**: 889 non-null → only **2 missing**. Negligible — can be filled with the mode (Southampton).

The dataset uses **28.7 KB** of memory — tiny. No performance concerns.


### C4 — `df.describe()` — Summary Statistics for Numeric Columns

In [ ]:
df.describe()

**Interpretation:**  
- **`Age`**: Mean ≈ 29.7, Median (50%) ≈ 28. Close — Age is roughly symmetric, though slightly right-skewed. Min = 0.42 (an infant), Max = 80.
- **`Fare`**: Mean ≈ 32.2, Median ≈ 14.5 — very different! This signals *right skew* caused by a small number of very expensive tickets (max = £512.33). The Fare distribution will need log-transformation before modelling.
- **`SibSp`** and **`Parch`**: Most passengers travelled alone (medians of 0). Max SibSp = 8 suggests large families.
- **`Pclass`**: Range 1–3 as expected. Mean ≈ 2.3 means more passengers were in 3rd class.

Connecting to C3: the Age mean (29.7) is slightly biased upward because the 177 missing values are not random — younger passengers and those in 3rd class were more likely to have missing age records.


### C5 — `df.isnull().sum()` — Exact Missing Value Counts

In [ ]:
# Absolute counts
print("=== Missing Value Counts ===")
print(df.isnull().sum())

print("\n=== Missing Value Percentages ===")
print((df.isnull().sum() / len(df) * 100).round(1))


**Interpretation:**  
- **`Cabin`** (77.1% missing): Drop this column or engineer a binary feature `has_cabin` (1 if cabin info exists, 0 if not). Cabin info correlates with wealth and therefore survival.
- **`Age`** (19.9% missing): Impute with median (robust to outliers) or use a predictive model. Do NOT use mean because of the slight skew.
- **`Embarked`** (0.2% missing): Fill 2 rows with the mode — 'S' (Southampton), which accounts for ~72% of passengers.

All other columns are complete. Connecting to C4: the Age mean we saw earlier is computed only on the 714 non-missing values, which may be biased.


### C6 — `df['Survived'].value_counts()` — Target Class Distribution

In [ ]:
print("=== Survived Value Counts ===")
print(df['Survived'].value_counts())
print()
print("=== As Percentages ===")
print(df['Survived'].value_counts(normalize=True).round(3) * 100)


**Interpretation:**  
- **549 passengers did not survive (61.6%)** and **342 survived (38.4%)**.
- This is a **class imbalance**. A naive model that always predicts "did not survive" would achieve 61.6% accuracy — but this is meaningless because it learns nothing.
- When modelling this dataset, we must use metrics like **F1-score**, **Precision-Recall**, or **ROC-AUC** instead of raw accuracy.
- Strategies to address imbalance: oversampling the minority class (SMOTE), undersampling the majority, or class-weighted loss functions.


### C7 — `df.duplicated().sum()` — Duplicate Row Detection

In [ ]:
print(f"Number of duplicate rows: {df.duplicated().sum()}")

# Also show any duplicate rows (if they exist)
dupes = df[df.duplicated()]
print(f"\nDuplicate rows preview:")
print(dupes if len(dupes) > 0 else "No duplicate rows found.")


**Interpretation:**  
**Zero duplicate rows** — the Titanic dataset is clean in this regard. Each row represents a unique passenger.

This matters because duplicates silently inflate statistics: if a high-fare passenger appeared twice, the mean Fare would be artificially increased, and any groupby aggregations would be biased. In e-commerce or transaction datasets, duplicates are extremely common and must be removed before any analysis.


---
## Section 3 — 20+ Additional EDA Commands


### EDA 1 — `df.head(10)` — First 10 Rows

In [ ]:
df.head(10)

**Interpretation:**  
The first 10 rows reveal the general structure immediately. I can see that `Cabin` is mostly `NaN` even in the opening rows — confirming the 77% missingness we found in C3. `Name` contains titles (Mr., Mrs., Miss.) which could be extracted as a new feature. `Ticket` values look alphanumeric with no clear pattern at first glance.


### EDA 2 — `df.tail(5)` — Last 5 Rows

In [ ]:
df.tail(5)

**Interpretation:**  
The last 5 rows look structurally similar to the first — no obvious data-loading truncation errors (like repeated headers or summary rows that sometimes appear at the end of exported CSVs). This is a well-formed file. `Cabin` is again missing for these rows.


### EDA 3 — `df.sample(15)` — 15 Random Rows

In [ ]:
df.sample(15, random_state=42)

**Interpretation:**  
Sampling random rows prevents the bias of only ever seeing the "beginning" of the data. This view confirms variety across `Pclass` (1, 2, and 3 all appear), `Sex`, and `Survived`. I can see a range of ages including at least one child. The `Fare` column shows large variation in this small random sample — confirming the extreme spread C4 revealed.


### EDA 4 — `df.columns.tolist()` — All Column Names

In [ ]:
print(df.columns.tolist())

**Interpretation:**  
All 12 column names are clean — no hidden spaces (like `'Age '` vs `'Age'`), no special characters, and no inconsistent capitalisation within the same dataset load. This matters because `df['Age']` and `df['Age ']` are different keys in Pandas, and a trailing space causes confusing KeyError bugs that are hard to spot visually. We're clean here.


### EDA 5 — `df.nunique()` — Unique Value Count Per Column

In [ ]:
df.nunique()

**Interpretation:**  
- **`PassengerId`** and **`Name`** each have 891 unique values — they are ID-like columns and should never be used as model features.
- **`Ticket`**: 681 unique values — many tickets are shared (families/groups bought one ticket for multiple people).
- **`Cabin`**: 147 unique values from only 204 non-null entries — some cabins housed multiple passengers.
- **`Survived`**, **`Sex`**, **`Embarked`** have very few unique values (2–3) — these are true categorical columns.
- **`Pclass`** has 3 unique values — ordinal categorical.


### EDA 6 — `df['Sex'].unique()` — Distinct Values in Sex Column

In [ ]:
print("Sex unique values:", df['Sex'].unique())
print("Embarked unique values:", df['Embarked'].unique())
print("Pclass unique values:", df['Pclass'].unique())


**Interpretation:**  
- `Sex` has exactly two values: `'male'` and `'female'` — lowercase and consistent. No misspellings like `'Male'` or `'FEMALE'` that would cause silent errors in groupby operations.
- `Embarked` has `'S'`, `'C'`, `'Q'`, and `NaN` — three ports (Southampton, Cherbourg, Queenstown) plus the two missing values we already know about.
- `Pclass` is correctly limited to 1, 2, 3.

Always check categorical columns for unexpected values — a single `'femlae'` typo would create a third gender category silently.


### EDA 7 — `df.corr()` — Correlation Matrix

In [ ]:
corr = df.corr(numeric_only=True)
print(corr.round(2))


**Interpretation:**  
Key correlations with `Survived`:
- **`Fare`** (r = +0.26): Higher fare → higher survival. Makes sense — wealthier passengers had cabin access and better evacuation.
- **`Pclass`** (r = −0.34): Higher class number (lower class) → lower survival. Strongest numeric predictor.
- **`Age`** (r = −0.08): Weakly negative — older passengers slightly less likely to survive. Weak signal.
- **`SibSp`** and **`Parch`** are weakly correlated with each other (r ≈ 0.41) — families naturally travel together.

Note: correlation only captures **linear** relationships. Non-linear relationships (e.g., children had high survival regardless of class) won't appear here.


### EDA 8 — `df['Age'].hist()` — Age Distribution

In [ ]:
plt.figure(figsize=(8, 4))
df['Age'].hist(bins=20, color='steelblue', edgecolor='white')
plt.title('Age Distribution of Titanic Passengers')
plt.xlabel('Age')
plt.ylabel('Count')
plt.tight_layout()
plt.show()


**Interpretation:**  
The age distribution is roughly **right-skewed** with a peak around 20–30 years old. There is a visible bump for young children (0–10), reflecting families aboard. The tail extends to 80. The distribution is not perfectly normal — it has the characteristic shape of working-age travellers with a small but meaningful younger cohort. This slight skew means **median imputation is safer than mean imputation** for missing Age values.


### EDA 9 — `df.groupby('Sex').mean()` — Survival Rate by Gender

In [ ]:
df.groupby('Sex').mean(numeric_only=True).round(2)

**Interpretation:**  
This is one of the most revealing commands. The `Survived` column shows:
- **Female mean survival ≈ 0.74** (74% of women survived)
- **Male mean survival ≈ 0.19** (only 19% of men survived)

This massive gap reflects the "women and children first" evacuation policy. Sex is likely the **single most predictive feature** in this dataset. Also notice that females paid higher average fares and were in slightly higher classes — which compounds the survival difference.


### EDA 10 — `df['Pclass'].value_counts(normalize=True)` — Class Proportions

In [ ]:
print("=== Pclass Distribution (Proportions) ===")
print((df['Pclass'].value_counts(normalize=True) * 100).round(1))


**Interpretation:**  
- **3rd class**: 55.1% of passengers — the majority were in the lowest class.
- **1st class**: 24.2%
- **2nd class**: 20.7%

This means any model trained without considering class balance will be more heavily influenced by 3rd-class passenger patterns. Combined with C6 (survival imbalance), we have a nested imbalance problem — most non-survivors were 3rd-class passengers.


### EDA 11 — `df.select_dtypes(include='object')` — All Text/Categorical Columns

In [ ]:
text_cols = df.select_dtypes(include='object')
print("Text/Categorical columns:", text_cols.columns.tolist())
print(f"\nShape of text-only subset: {text_cols.shape}")
text_cols.head(5)


**Interpretation:**  
Five columns are of type `object`: `Name`, `Sex`, `Ticket`, `Cabin`, `Embarked`. These cannot be fed directly into most ML algorithms — they need encoding:
- `Sex` → Binary encode (0/1)
- `Embarked` → One-hot encode (3 dummy variables)
- `Name` → Extract title (Mr, Mrs, Miss, Master, etc.) then encode
- `Ticket` → Complex, may be dropped or hashed
- `Cabin` → Convert to binary `has_cabin` or extract deck letter

This command is useful for quickly isolating the columns that need the most preprocessing work.


### EDA 12 — `df['Name'].str.contains('Dr.').sum()` — Count Doctor Passengers

In [ ]:
dr_count = df['Name'].str.contains('Dr\.').sum()
master_count = df['Name'].str.contains('Master').sum()
miss_count = df['Name'].str.contains('Miss').sum()
rev_count = df['Name'].str.contains('Rev\.').sum()

print(f"Passengers with title 'Dr.'    : {dr_count}")
print(f"Passengers with title 'Master' : {master_count}")
print(f"Passengers with title 'Miss'   : {miss_count}")
print(f"Passengers with title 'Rev.'   : {rev_count}")


**Interpretation:**  
The `Name` column contains embedded titles that carry meaningful information:
- **'Master'** typically denotes male children — useful for identifying young boys whose survival rate was higher.
- **'Miss'** denotes unmarried females (usually younger).
- **'Dr.'** and **'Rev.'** are rare titles that could be grouped into a 'Rare' category.

Extracting titles from names is a well-known feature engineering step for this dataset. A simple regex like `df['Name'].str.extract(r', ([A-Za-z]+)\.')` would pull out all titles cleanly.


### EDA 13 — `df.sort_values('Fare', ascending=False).head(5)` — Top 5 Highest Fares

In [ ]:
df.sort_values('Fare', ascending=False).head(5)

**Interpretation:**  
The top fares are all above £200, with some reaching £512 — more than 35 times the median fare of ~£14. These are First Class passengers, and interestingly, several share the same ticket number, suggesting the fare is for the entire group/suite rather than per person. This is an important insight: **Fare may need to be divided by the number of people sharing a ticket** to get the true per-person cost. All five high-fare passengers survived — consistent with the Pclass-survival relationship.


### EDA 14 — `df[df['Age'] < 10]` — Filter: Child Passengers (Under 10)

In [ ]:
children = df[df['Age'] < 10]
print(f"Number of children under 10: {len(children)}")
print(f"\nSurvival rate among children under 10:")
print(children['Survived'].value_counts())
print(f"\nSurvival rate: {children['Survived'].mean():.1%}")


**Interpretation:**  
Children under 10 had a notably **higher survival rate** (~59%) compared to the overall rate of 38.4%. This confirms the "women and children first" protocol had real impact. However, not all children survived — 3rd-class children had far lower survival odds than 1st-class children. This interaction between age and class is something a simple correlation matrix (EDA 7) would miss entirely — illustrating why we need multiple EDA approaches.


### EDA 15 — `pd.pivot_table()` — Survival by Sex and Pclass

In [ ]:
pivot = pd.pivot_table(df, 
                      values='Survived', 
                      index='Sex', 
                      columns='Pclass', 
                      aggfunc='mean')
print("=== Survival Rate by Sex and Pclass ===")
print(pivot.round(2))


**Interpretation:**  
This pivot table is extremely revealing:

| | 1st Class | 2nd Class | 3rd Class |
|---|---|---|---|
| Female | 97% | 92% | 50% |
| Male | 37% | 16% | 13% |

Key insights:
1. **Being female in 1st class = 97% survival** — almost certain to survive.
2. **Being male in 3rd class = 13% survival** — almost certain to die.
3. Class mattered more for men than women — even 3rd-class women had a 50% survival rate, while 1st-class men only had 37%.
4. This interaction effect (`Sex × Pclass`) is more powerful than either variable alone.

This is exactly the kind of two-variable summary that `value_counts()` alone can't show.


### EDA 16 — `df.skew()` — Skewness of Numeric Columns

In [ ]:
print(df.skew(numeric_only=True).round(2))

**Interpretation:**  
- **`Fare`** has skewness of ~4.8 — highly right-skewed (a few extremely expensive tickets pull the mean far right). A log transformation is strongly recommended before modelling.
- **`SibSp`** and **`Parch`** are both positively skewed — most passengers travelled alone, with a long tail of large family groups.
- **`Age`** is mildly skewed (~0.39) — closer to normal, less transformation needed.
- Any feature with |skewness| > 1 typically benefits from log or Box-Cox transformation.


### EDA 17 — `df['Fare'].mode()` — Most Common Fare Value

In [ ]:
print("Most common Fare value(s):")
print(df['Fare'].mode())

print(f"\nMost common Embarked port:")
print(df['Embarked'].mode()[0])


**Interpretation:**  
- The most common Fare is **£8.05** — a 3rd-class standard fare. Many passengers paid exactly this amount, creating a spike at the low end of the Fare distribution that we saw in the histogram.
- The mode of `Embarked` is `'S'` (Southampton) — confirming our earlier plan to fill the 2 missing values with 'S'.
- Mode is the correct measure of "most common" for both skewed numeric data and categorical data. Mean or median would be misleading for `Embarked`.


### EDA 18 — `df.memory_usage()` — Memory Footprint Per Column

In [ ]:
print("=== Memory Usage Per Column (bytes) ===")
print(df.memory_usage(deep=True))
print(f"\nTotal memory: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")


**Interpretation:**  
The total dataset uses around **350–400 KB** in memory (deep=True counts the actual string content in object columns). This is tiny — no memory optimisation needed here. In production datasets with millions of rows, memory profiling is critical: switching `int64` to `int32` or `float64` to `float32` can halve memory usage. `object` columns (strings) are the most memory-hungry — they explain why `Name` and `Cabin` consume more bytes than numeric columns despite having the same number of rows.


### EDA 19 — `df.kurt()` — Kurtosis of Numeric Columns

In [ ]:
print(df.kurt(numeric_only=True).round(2))

**Interpretation:**  
- **`Fare`** has extremely high kurtosis (~28) — a *leptokurtic* distribution. This means it has a very sharp central peak and very heavy tails (the extreme fares of £512). This further justifies log-transformation.
- **`SibSp`** and **`Parch`** also show high kurtosis — most values are 0, creating a very sharp spike at zero with sparse high values.
- **`Age`** kurtosis is near 0 (mesokurtic) — close to a normal distribution's shape.
- High kurtosis means outliers are more extreme and more frequent than a normal distribution would predict.


### EDA 20 — `df.cov()` — Covariance Matrix

In [ ]:
df.cov(numeric_only=True).round(2)

**Interpretation:**  
The covariance matrix shows the **raw co-variation** between numeric columns — but since it's not normalised, large-valued columns (like `Fare`) dominate. Key takeaways:
- `Fare` has a large covariance with itself (variance = ~2470) — this reflects its extreme spread (max 512, most values under 50).
- `Fare` and `Pclass` have a **negative covariance** — as Fare goes up, Pclass number goes down (i.e., higher fare = higher class). Makes sense.
- Unlike correlation, covariance values depend on the scale of measurement, making them harder to compare across columns. That's why we use correlation (EDA 7) for pattern spotting and covariance mainly for mathematical operations (like PCA).


### EDA 21 — Family Size Feature Engineering

In [ ]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1  # +1 for the passenger themselves

print("Family Size Distribution:")
print(df['FamilySize'].value_counts().sort_index())

print("\nSurvival Rate by Family Size:")
print(df.groupby('FamilySize')['Survived'].mean().round(2))


**Interpretation:**  
Creating `FamilySize = SibSp + Parch + 1` combines two weak features into a stronger one. The survival-by-family-size breakdown reveals:
- **Solo travellers** (FamilySize=1): ~30% survival — travelling alone was disadvantageous.
- **Small families** (2–4): ~50–58% survival — best odds. Likely helped each other to lifeboats.
- **Large families** (5+): survival plummets — large families had trouble staying together and securing enough lifeboat spots.

This U-shaped relationship (non-linear!) would not appear in a linear correlation. Feature engineering turns domain knowledge into model signal.


### EDA 22 — Engineering a Binary `has_cabin` Feature

In [ ]:
df['has_cabin'] = df['Cabin'].notna().astype(int)

print("has_cabin distribution:")
print(df['has_cabin'].value_counts())

print("\nSurvival rate by cabin availability:")
print(df.groupby('has_cabin')['Survived'].mean().round(3))


**Interpretation:**  
Rather than dropping the `Cabin` column entirely (losing information) or keeping it as 77%-missing (causing errors), we engineer a binary `has_cabin` flag:
- **has_cabin = 0** (no cabin info): ~30% survival rate
- **has_cabin = 1** (cabin info known): ~67% survival rate

This 2× difference shows that merely *having a documented cabin* is strongly predictive of survival — likely because cabin records were better kept for 1st-class passengers, and 1st-class passengers had better evacuation access. This is a practical example of how to extract signal from missing data rather than discarding it.


---
## Section 4 — 3 Key Findings in Plain English

---

### 🔍 Finding 1: Gender Was the Strongest Survival Predictor
Women survived at a rate of **74%** while men survived at only **19%** — a nearly 4× difference. The "women and children first" policy was strictly followed. When combined with passenger class (EDA 15), the effect is even starker: 97% of first-class women survived, while only 13% of third-class men did. If I were to build a machine learning model, `Sex` would almost certainly be the most important feature.

---

### 🔍 Finding 2: Fare Is Extremely Skewed and Needs Transformation
The `Fare` column has a mean of ~£32 but a median of only ~£14.5, with a maximum of £512 (EDA 4). Its skewness is ~4.8 and kurtosis ~28 (EDA 16, EDA 19). A small number of extremely expensive ticket prices — likely representing luxury suites booked for a group — pull the average far from what a typical passenger paid. Before using Fare in any model, a **log transformation** (`np.log1p(df['Fare'])`) is essential to bring it closer to a normal distribution.

---

### 🔍 Finding 3: Missing Data Is Non-Random and Tells a Story
The 77% missing rate in `Cabin` is not random — it correlates directly with passenger class. First-class passengers had documented cabin assignments; third-class passengers mostly did not. This means imputing or dropping `Cabin` naively would silently discard a useful signal. By engineering a binary `has_cabin` flag (EDA 22), we turned a 77%-missing column into a feature with a clear survival signal: **passengers with cabin records survived at 67%** vs. **30% for those without**. Always ask *why* data is missing before deciding how to handle it.


---
*Assignment completed for Week 1 of the Data Science Internship Program*  
*Mentor: Laiba Sattar | Dataset: Titanic Passenger Data*
